## The boot chain — GRUB → initramfs → systemd

Now the detail behind the hook: four well-defined stages between power-on and a login prompt. Knowing them is what lets you fix a machine that won't boot.

**1. BIOS / UEFI firmware.** On power-on the motherboard **firmware** runs first (you don't choose it), does the POST hardware check, and finds a boot device. Legacy **BIOS** loads the first 512 bytes (the MBR); modern **UEFI** reads the EFI System Partition (usually `/boot/efi`) and runs a `.efi` bootloader. Tell which you're on: `[ -d /sys/firmware/efi ] && echo UEFI || echo BIOS`.

**2. GRUB.** The bootloader on virtually all Linux systems presents a menu, then loads the chosen kernel. ⚠️ **Don't edit `/boot/grub/grub.cfg` directly** — it's *generated* from `/etc/default/grub` plus `/etc/grub.d/`; edit those and run **`sudo update-grub`** (Debian) / `grub2-mkconfig` (RHEL). Common knobs: `GRUB_TIMEOUT`, `GRUB_CMDLINE_LINUX`. At the menu you can press **`e`** to edit an entry for one boot — the recovery escape hatch.

**3. Kernel + initramfs.** GRUB loads two files: the **kernel** (`vmlinuz`) and the **initramfs** (`initrd.img`) — a tiny in-memory filesystem holding just enough drivers and tools to find and mount the *real* root. It exists because `/` might sit on LVM on an encrypted disk on RAID; the kernel needs those tools *before* it can mount `/`. Once `/` is mounted, the initramfs is discarded. Rebuild it with **`update-initramfs -u`** (Debian) / `dracut -f` (RHEL).

**4. systemd.** The kernel runs `/sbin/init` — a symlink to **systemd** — as PID 1, which starts everything and reaches the default target: a login prompt. And that's the fourth stage the rest of the module lives in.
